In [17]:
import os
import json
import base64
import requests

try:
    from dotenv import load_dotenv

    load_dotenv()
except Exception:
    pass

try:
    from openai import OpenAI
except Exception:
    OpenAI = None

HOST_IP = os.getenv("HOST_IP", "192.168.1.13")

LLM_PORT = int(os.getenv("LLM_PORT", "10000"))
VLM_PORT = int(os.getenv("VLM_PORT", str(LLM_PORT)))
EMBEDDING_PORT = int(os.getenv("EMBEDDING_PORT", "10004"))
RERANKER_PORT = int(os.getenv("RERANKER_PORT", "10005"))
ASR_PORT = int(os.getenv("ASR_PORT", "10019"))

API_KEY = os.getenv("API_KEY", "xtsk")

def auth_headers():
    return {
        "Content-Type": "application/json",
        "Authorization": f"Bearer {API_KEY}",
    }

def post_json(url: str, payload: dict, timeout: int = 300):
    resp = requests.post(url, headers=auth_headers(), json=payload, timeout=timeout)
    resp.raise_for_status()
    return resp.json()

def openai_client(port: int):
    if OpenAI is None:
        return None
    base_url = f"http://{HOST_IP}:{port}/v1"
    return OpenAI(base_url=base_url, api_key=API_KEY)

def iter_sse_lines(response):
    for raw in response.iter_lines(decode_unicode=True):
        if not raw:
            continue
        if raw.startswith("data: "):
            yield raw[len("data: "):].strip()


In [22]:
print("LLM - openai")
client = openai_client(LLM_PORT)
if client is None:
    print("openai package not available")
else:
    resp = client.chat.completions.create(
        model="LLM",
        messages=[
            {"role": "system", "content": "You are a helpful assistant."},
            {"role": "user", "content": "用一句话解释什么是向量检索"},
        ],
        temperature=0.2,
    )
    print(resp.choices[0].message.content)

print("LLM - openai streaming")
if client is None:
    print("openai package not available")
else:
    stream = client.chat.completions.create(
        model="LLM",
        messages=[
            {"role": "system", "content": "You are a helpful assistant."},
            {"role": "user", "content": "用一句话解释什么是向量检索"},
        ],
        temperature=0.2,
        stream=True,
    )
    for event in stream:
        delta = event.choices[0].delta
        if delta and delta.content:
            print(delta.content, end="", flush=True)
    print()

print("LLM - requests")
url = f"http://{HOST_IP}:{LLM_PORT}/v1/chat/completions"
payload = {
    "model": "LLM",
    "messages": [
        {"role": "system", "content": "You are a helpful assistant."},
        {"role": "user", "content": "用一句话解释什么是向量检索"},
    ],
    "temperature": 0.2,
}

data = post_json(url, payload)
print(data["choices"][0]["message"]["content"])

print("LLM - requests streaming")
payload_stream = dict(payload)
payload_stream["stream"] = True
with requests.post(url, headers=auth_headers(), json=payload_stream, stream=True, timeout=300) as response:
    response.raise_for_status()
    for sse in iter_sse_lines(response):
        if sse == "[DONE]":
            break
        event_data = json.loads(sse)
        delta = event_data.get("choices", [{}])[0].get("delta", {})
        content = delta.get("content")
        if content:
            print(content, end="", flush=True)
print()


LLM - openai


InternalServerError: Error code: 502

In [ ]:
local_image = "test.jpg"
if not os.path.exists(local_image):
    raise FileNotFoundError(f"image file not found: {local_image}")

with open(local_image, "rb") as f:
    image_b64 = base64.b64encode(f.read()).decode("utf-8")

print("VLM - openai")
client = openai_client(VLM_PORT)
if client is None:
    print("openai package not available")
else:
    resp = client.chat.completions.create(
        model="VLM",
        messages=[
            {
                "role": "user",
                "content": [
                    {"type": "text", "text": "请描述图片里最主要的内容。"},
                    {
                        "type": "image_url",
                        "image_url": {"url": f"data:image/jpeg;base64,{image_b64}"},
                    },
                ],
            }
        ],
        temperature=0.2,
    )
    print(resp.choices[0].message.content)

print("VLM - requests")
url = f"http://{HOST_IP}:{VLM_PORT}/v1/chat/completions"

payload = {
    "model": "VLM",
    "messages": [
        {
            "role": "user",
            "content": [
                {"type": "text", "text": "请描述图片里最主要的内容。"},
                {
                    "type": "image_url",
                    "image_url": {"url": f"data:image/jpeg;base64,{image_b64}"},
                },
            ],
        }
    ],
    "temperature": 0.2,
}

data = post_json(url, payload)
print(data["choices"][0]["message"]["content"])


language English<asr_text>Hmm. Oh yeah, yeah. He wasn't even that big when I started listening to him, but and his solo music didn't do overly well. But he did very well when he started writing for other people.


In [ ]:
local_audio = "di-test-sr-1.wav"
if not os.path.exists(local_audio):
    raise FileNotFoundError(f"audio file not found: {local_audio}")

with open(local_audio, "rb") as f:
    audio_b64 = base64.b64encode(f.read()).decode("utf-8")

messages = [
    {
        "role": "user",
        "content": [
            {
                "type": "audio_url",
                "audio_url": {"url": f"data:audio/wav;base64,{audio_b64}"},
            }
        ],
    }
]

print("ASR - openai")
client = openai_client(ASR_PORT)
if client is None:
    print("openai package not available")
else:
    resp = client.chat.completions.create(
        model="ASR",
        messages=messages,
    )
    print(resp.choices[0].message.content)

print("ASR - openai streaming")
if client is None:
    print("openai package not available")
else:
    stream = client.chat.completions.create(
        model="ASR",
        messages=messages,
        stream=True,
    )
    for event in stream:
        delta = event.choices[0].delta
        if delta and delta.content:
            print(delta.content, end="", flush=True)
    print()

print("ASR - requests")
url = f"http://{HOST_IP}:{ASR_PORT}/v1/chat/completions"
payload = {
    "model": "ASR",
    "messages": messages,
}

data = post_json(url, payload)
print(data["choices"][0]["message"]["content"])

print("ASR - requests streaming")
payload_stream = dict(payload)
payload_stream["stream"] = True
with requests.post(url, headers=auth_headers(), json=payload_stream, stream=True, timeout=300) as response:
    response.raise_for_status()
    for sse in iter_sse_lines(response):
        if sse == "[DONE]":
            break
        event_data = json.loads(sse)
        delta = event_data.get("choices", [{}])[0].get("delta", {})
        content = delta.get("content")
        if content:
            print(content, end="", flush=True)
print()


In [ ]:
texts = [
    "向量数据库适合存什么？",
    "embedding 有什么用？",
]

print("EMBEDDING - openai")
client = openai_client(EMBEDDING_PORT)
if client is None:
    print("openai package not available")
else:
    resp = client.embeddings.create(
        model="EMBEDDING",
        input=texts,
    )
    print("dim =", len(resp.data[0].embedding))
    print("first 20 =", resp.data[0].embedding[:20])

print("EMBEDDING - requests")
url = f"http://{HOST_IP}:{EMBEDDING_PORT}/v1/embeddings"

payload = {
    "model": "EMBEDDING",
    "input": texts,
}

data = post_json(url, payload)
print("dim =", len(data["data"][0]["embedding"]))
print("first 20 =", data["data"][0]["embedding"][:20])


In [21]:
query = "如何办理退税？"
documents = [
    "你可以在 APP 里提交申请。",
    "退税需要满足一定条件，并准备相关材料。",
    "今天天气不错。",
]

print("RERANKER - openai")
client = openai_client(RERANKER_PORT)
if client is None:
    print("openai package not available")
else:
    try:
        resp = client.chat.completions.create(
            model="RERANKER",
            messages=[
                {
                    "role": "user",
                    "content": json.dumps(
                        {"query": query, "documents": documents, "top_n": 4},
                        ensure_ascii=False,
                    ),
                }
            ],
        )
        print(resp.choices[0].message.content)
    except Exception as e:
        print("openai rerank call failed:", e)

print("RERANKER - requests")
url = f"http://{HOST_IP}:{RERANKER_PORT}/rerank"

payload = {
    "model": "RERANKER",
    "query": query,
    "documents": documents,
    "top_n": 4,
}

try:
    data = post_json(url, payload)
    print(data)
except Exception as e:
    print("requests rerank failed:", e)


RERANKER - openai
openai rerank call failed: Error code: 502
RERANKER - requests
{'error': {'message': 'The model does not support Rerank (Score) API', 'type': 'BadRequestError', 'param': None, 'code': 400}}
